# Filtering in the Spatial Domain — Problem Statement


## Context

Spatial filtering computes each output pixel from a local neighborhood. Reliable filtering requires explicit kernels, border handling, noise assumptions, numerical safety, and quantitative/visual validation.


## Problem Statement

Develop a reproducible spatial-domain filtering workflow that explains convolution/correlation, smoothing, denoising, sharpening, gradients, border effects, color handling, and filter selection from first principles.


## Inputs and Fixed Parameters

Use module-local images under `../data/`, repository-relative paths, explicit kernels and border modes, floating-point intermediate arithmetic, and generated figures under `../outputs/figures/`.


## 1. Data and Output Paths

The notebook locates the laboratory directory automatically by searching upward for both `data/` and `notebooks/`.

This makes execution robust whether VS Code starts the notebook from the repository root, the lab root, or the notebook directory.


## 2. What Is Spatial Filtering?

A **point operation** transforms one pixel using only its own value:

$$
g(x,y)=T(f(x,y))
$$

A **spatial neighborhood operation** uses nearby pixels:

$$
g(x,y)=T\left(\text{neighborhood around }(x,y)\right)
$$

This is the fundamental difference between the previous image-transformation laboratory and the present filtering laboratory.

Spatial filters can:

- suppress noise;
- blur small structures;
- preserve or destroy edges;
- emphasize rapid intensity changes;
- sharpen an image;
- estimate local derivatives.


## 3. Kernel Anatomy

A **kernel** (also called a mask or filter) is a small matrix of weights.

For a linear $3\times3$ filter:

$$
K=
\begin{bmatrix}
k_{-1,-1} & k_{0,-1} & k_{1,-1}\\
k_{-1,0}  & k_{0,0}  & k_{1,0}\\
k_{-1,1}  & k_{0,1}  & k_{1,1}
\end{bmatrix}
$$

Important kernel properties include:

- **size** — e.g. 3×3, 5×5, 9×9;
- **anchor / center** — location aligned with the current pixel;
- **weights** — determine how neighbors contribute;
- **sum of weights** — often controls response to constant regions;
- **symmetry** — important for smoothing and derivative behavior.

A normalized smoothing kernel usually has weights summing to 1.


## 4. Correlation vs Convolution

Correlation and convolution both slide a kernel across an image.

The difference is the kernel orientation.

### Correlation

For correlation, the kernel is used as written.

### Convolution

For convolution, the kernel is flipped horizontally and vertically before the sliding operation.

In 2-D:

$$
K_{\mathrm{conv}}(i,j)=K(-i,-j)
$$

If a kernel is symmetric, correlation and convolution produce the same result.

If it is asymmetric, they generally differ.


## 5. Convolution from First Principles

Before using library functions, we implement a small educational convolution routine.

The objective is not speed. The objective is to understand the repeated operations:

```text
pad image
    ↓
extract neighborhood
    ↓
multiply by flipped kernel
    ↓
sum
    ↓
store output pixel
```


## 6. Border Handling

At the image border, part of the neighborhood lies outside the array.

A filtering algorithm must decide what values exist beyond the image.

Common strategies include:

- `constant` → fill with a fixed value, often 0;
- `nearest` → repeat the nearest border pixel;
- `reflect` → mirror the image around the edge;
- `mirror` → a related reflection convention;
- `wrap` → continue from the opposite side.

The border rule can change the numerical result.


## 7. Mean / Box Filtering

The $3\times3$ mean filter is:

$$
K=
\frac{1}{9}
\begin{bmatrix}
1&1&1\\
1&1&1\\
1&1&1
\end{bmatrix}
$$

Each output pixel is the arithmetic mean of its neighborhood.

The mean filter reduces local fluctuations, but it does not know whether a variation is noise or a real edge.

Therefore:

```text
larger averaging neighborhood
    → stronger smoothing
    → stronger edge/detail loss
```


## 8. Gaussian Filtering

A Gaussian filter gives larger weight to nearby pixels and smaller weight to distant pixels.

The continuous 2-D Gaussian is:

$$
G(x,y)=
\frac{1}{2\pi\sigma^2}
\exp\left(
-\frac{x^2+y^2}{2\sigma^2}
\right)
$$

The parameter $\sigma$ controls the spatial spread:

- small $\sigma$ → weak smoothing;
- large $\sigma$ → stronger smoothing.

Unlike a box filter, the weights vary smoothly with distance.


## 9. Why Noise Type Matters

The same filter should not be selected blindly for every degradation.

The provided Einstein images let us compare three important noise types:

- Gaussian;
- salt-and-pepper;
- speckle.

Their visual structure is different, so their preferred filters can also differ.


## 10. Median Filtering

The median filter is **nonlinear**.

For each neighborhood:

1. collect all pixel values;
2. sort them;
3. select the middle value;
4. assign that median to the output pixel.

Example neighborhood values:

```text
[20, 21, 20,
 22, 255, 19,
 20, 21, 20]
```

The value `255` is an impulse outlier.

The median remains near the normal neighborhood values instead of being pulled strongly upward.


## 11. Bilateral Filtering

A bilateral filter smooths pixels using two notions of similarity:

1. **spatial similarity** — nearby pixels matter more;
2. **intensity similarity** — pixels with similar intensity matter more.

A simplified bilateral weight between a center pixel $p$ and a neighbor $q$ is:

$$
w(p,q)
=
\exp\left(
-\frac{\|p-q\|^2}{2\sigma_s^2}
\right)
\exp\left(
-\frac{|I_p-I_q|^2}{2\sigma_r^2}
\right)
$$

where:

- $\sigma_s$ controls spatial distance;
- $\sigma_r$ controls intensity/range similarity.

Because pixels across a strong edge have very different intensities, their contribution can be reduced.


## 12. Quantitative Denoising Metrics

Visual inspection is essential, but quantitative metrics help compare outputs reproducibly.

For reference image $R$ and test image $T$:

### MAE

$$
\mathrm{MAE}
=
\frac{1}{N}\sum |R-T|
$$

### MSE

$$
\mathrm{MSE}
=
\frac{1}{N}\sum (R-T)^2
$$

### RMSE

$$
\mathrm{RMSE}=\sqrt{\mathrm{MSE}}
$$

### PSNR

For 8-bit images:

$$
\mathrm{PSNR}
=
10\log_{10}
\left(
\frac{255^2}{\mathrm{MSE}}
\right)
$$

Higher PSNR normally means lower pixel-wise error.


## 13. Edge Preservation as a Secondary Check

One simple way to inspect structural preservation is to compare gradient magnitude.

This is not a universal perceptual-quality metric, but it gives useful intuition about whether smoothing has weakened edges.


## 14. Sharpening with the Laplacian

Smoothing suppresses high local variation.

Sharpening does the opposite: it emphasizes rapid intensity changes.

The continuous Laplacian is:

$$
\nabla^2 f
=
\frac{\partial^2 f}{\partial x^2}
+
\frac{\partial^2 f}{\partial y^2}
$$

A common discrete 4-neighbor Laplacian kernel is:

$$
\begin{bmatrix}
0&1&0\\
1&-4&1\\
0&1&0
\end{bmatrix}
$$

Because Laplacian sign conventions differ between implementations, the sharpening formula must be checked carefully.


## 15. Unsharp Masking and High-Boost Filtering

Unsharp masking first creates a blurred version of the image.

The detail mask is:

$$
m=f-f_{\mathrm{blur}}
$$

Then the sharpened image is:

$$
g=f+k\,m
$$

where $k$ controls sharpening strength.

- $k=1$ → classical unsharp masking;
- $k>1$ → stronger high-boost sharpening.


## 16. First Derivatives and Image Gradients

Edges correspond to rapid intensity variation.

The image gradient contains horizontal and vertical derivatives:

$$
\nabla f=
\begin{bmatrix}
G_x\\
G_y
\end{bmatrix}
$$

The gradient magnitude is:

$$
|\nabla f|
=
\sqrt{G_x^2+G_y^2}
$$

The gradient orientation is:

$$
\theta
=
\operatorname{atan2}(G_y,G_x)
$$

The Sobel operator combines differentiation with a small amount of local smoothing.


## 17. Sobel vs Prewitt vs Scharr

Several derivative operators approximate image gradients.

### Prewitt

Uses simple derivative and smoothing weights.

### Sobel

Gives larger weight to the center row/column and is extremely common.

### Scharr

Uses coefficients designed to improve rotational symmetry for a 3×3 derivative operator.

No operator is universally best for every problem.


## 18. Border Effects on a Real Image

Large kernels make border behavior easier to see.

We compare several padding modes using a 15×15 averaging filter.


## 19. Filtering RGB Images

A color image has shape:

```text
(H, W, 3)
```

A spatial filter should normally act over the two spatial dimensions while preserving the channel dimension.

For a Gaussian filter in SciPy, this can be expressed using:

```python
sigma=(sigma_y, sigma_x, 0)
```

The zero prevents smoothing across the channel axis.


## 20. Numerical Safety

Filtering often produces floating-point values or signed derivative responses.

Important rules:

1. Convert to floating point before operations that may become negative or exceed 255.
2. Do not immediately cast derivative images to `uint8`.
3. Clip only when producing a display/storage image that requires a bounded range.
4. Keep signed responses when the sign contains information.
5. Normalize kernels deliberately rather than automatically.


## 21. Choosing a Filter

A useful first decision table is:

| Situation | Reasonable first choice | Why |
|---|---|---|
| Mild additive Gaussian noise | Gaussian filter | smooth weighted averaging |
| Random impulse/salt-and-pepper noise | Median filter | rejects isolated extreme values |
| Noise with important edges | Bilateral filter | weights both distance and intensity similarity |
| General simple smoothing | Mean or Gaussian | simple neighborhood averaging |
| Blur requiring local contrast enhancement | Unsharp mask / Laplacian | emphasizes high local variation |
| Edge/gradient estimation | Sobel / Scharr / Prewitt | approximates spatial derivatives |

This table is a starting point, not a substitute for validation.


## 22. Standard Spatial-Filtering Workflow

A robust workflow is:

```text
1. Inspect image
    ↓
2. Identify degradation or objective
    ↓
3. Check dtype and intensity range
    ↓
4. Choose filter family
    ↓
5. Choose border handling
    ↓
6. Start with conservative parameters
    ↓
7. Apply filter in floating point when needed
    ↓
8. Compare visually
    ↓
9. Compare numerically if reference exists
    ↓
10. Check edge/detail preservation
    ↓
11. Tune parameters
    ↓
12. Validate and save result
```


## 23. Validation Checks

The following assertions verify key mathematical and implementation assumptions.


## Completion Criterion

The Implementation notebook must execute end-to-end, generate the required spatial-filtering diagnostics, preserve valid numerical ranges/dtypes, and pass its validation checks.
